In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import json
from collections import Counter
import numpy as np
import pandas as pd
import seaborn as sns
import spacy
import re
import pycountry
import sys 
sys.path.append('/home/lhasbini/como_school/como_project4/src/')
from text_processing_functions import *
from LLM_functions import *
from plot_functions import *
import copy as cp
from random import randrange, randint
import ast
import geopy as gpy
import itertools
import time

ERROR 1: PROJ: proj_create_from_database: Open of /home/lhasbini/.conda/envs/dev/share/proj failed


In [15]:
###### FILE PATHS 
fig_path = '/home/lhasbini/como_school/figure/'
file_path_save = '/scratchx/lhasbini/como_school/'
json_file = '/scratchx/lhasbini/como_school/filtered_report_types_nat_hazards_summary-header.json'

##### Open and read the JSON file
with open(json_file, 'r') as json_file:
    filtered_reports = json.load(json_file)

#### EXAMPLE OF LABELLED REPORTS
df_labelled = pd.read_csv(file_path_save+'labelled_example_haz-subtype-emdat_laura.csv')
df_labelled = df_labelled.replace({np.nan: None})

In [16]:
# Convert the labelled examples to chat format
df_labelled_chat = convert_labelled_chat_format(df_labelled)
df_labelled_chat.to_csv(file_path_save+'labelled_example_haz-subtype-emdat_chat-format_laura.csv', encoding='utf-8', index=False)

In [19]:
df_labelled_chat

,hazardType,hazardSubtypes,country,region,city,locationAnnotation,startYear,startMonth,startDay,endYear,endMonth,endDay,hazardName,appealCode,Country
0,Flood,None,Algeria,None,Bchar,"The most affected areas include Bchar, Elbayad...",2024.0,9.0,5.0,2024.0,9.0,8.0,None,MDRDZ011,DZA
1,Flood,None,Algeria,None,Elbayadh,"The most affected areas include Bchar, Elbayad...",2024.0,9.0,5.0,2024.0,9.0,8.0,None,MDRDZ011,DZA
2,Flood,None,Algeria,None,Beni Abbes,"The most affected areas include Bchar, Elbayad...",2024.0,9.0,5.0,2024.0,9.0,8.0,None,MDRDZ011,DZA
3,Flood,None,Algeria,None,Tamanrasset,"The most affected areas include Bchar, Elbayad...",2024.0,9.0,5.0,2024.0,9.0,8.0,None,MDRDZ011,DZA
4,Flood,None,Algeria,None,Tiaret,"The most affected areas include Bchar, Elbayad...",2024.0,9.0,5.0,2024.0,9.0,8.0,None,MDRDZ011,DZA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
260,Flood,None,Brazil,None,Medeiros Neto,The main affected areas in the far south are i...,2021.0,11.0,None,2022.0,5.0,None,None,MDRBR010,BRA
261,Flood,None,Brazil,None,Petrópolis,"In mid-February and early March, further rains...",2021.0,11.0,None,2022.0,5.0,None,None,MDRBR010,BRA
262,Flood,None,Brazil,None,Angra dos Reis,"In mid-February and early March, further rains...",2021.0,11.0,None,2022.0,5.0,None,None,MDRBR010,BRA
263,Flood,None,Brazil,None,Paraty,"In mid-February and early March, further rains...",2021.0,11.0,None,2022.0,5.0,None,None,MDRBR010,BRA


In [39]:
# Add Geolocation and spatial information (region from cities)
from geopy.extra.rate_limiter import RateLimiter
from rapidfuzz.distance import Levenshtein

geolocator = gpy.geocoders.Nominatim(user_agent='laura.hasbini@lsce.ipsl.fr')
# geocode = RateLimiter(geolocator.geocode, min_delay_seconds=20)

In [55]:
row_test = df_labelled_chat.loc[46]
country = row_test["country"]
if row_test.city:
    finest_loc_id = "city"
    finest_loc_vals = row_test.city
elif row_test.region:
    finest_loc_id = "region"
    finest_loc_vals = row_test.region
else:
    finest_loc_id = None
    print("No location information for response: ", response_row.appealCode)


if finest_loc_id:
    # nominatim_query = {
    #     "country": country,
    #     finest_loc_id: finest_loc_vals
    # }
    nominatim_query=finest_loc_vals+", "+country
    nominatim_result = geolocator.geocode(nominatim_query, exactly_one=True, language="en")
    if nominatim_result is None:
        print("No results for query: ", nominatim_query)
    else:      
        ## Double check that the known information is similar to the one found 
        similarity = Levenshtein.normalized_similarity(finest_loc_vals, nominatim_result.raw['name'])
        if similarity > 0.8 : 
            print("Found location "+nominatim_result.raw['name']+" for "+finest_loc_vals+", similarity = "+str(similarity))
            row_test["latitude"] = nominatim_result.latitude
            row_test["longitude"] = nominatim_result.longitude
            
            #Find in case the location is a city
            if finest_loc_id== "city" : 
                #Reverse geocoding to get region/state name 
                location_details = geolocator.reverse((nominatim_result.latitude, nominatim_result.longitude), exactly_one=True, addressdetails=True)
                address = location_details.raw.get("address", {})
                region = address.get("region")
                if region is None : 
                    region = address.get("state")
                row_test["region"] = region            
        else : 
            print(nominatim_result.raw)
            print("Found location "+nominatim_result.raw['name']+" for "+finest_loc_vals+", similarity = "+str(similarity))
print(row_test)

Found location Mokolo for Mokolo, similarity = 1.0
hazardType                                                        Flood
hazardSubtypes                        ['flash flood', 'riverine flood']
country                                                        Cameroon
region                                                     Extrême-Nord
city                                                             Mokolo
locationAnnotation    Notably, in the Diamar division, where Ndoukou...
startYear                                                        2024.0
startMonth                                                          8.0
startDay                                                           10.0
endYear                                                          2024.0
endMonth                                                            8.0
endDay                                                             28.0
hazardName                                                          NaN
appealCode   

In [27]:
def rotated_levenshtein_similarity(str1, str2):
    """Compute the best Levenshtein similarity considering all rotations of words."""
    words1, words2 = str1.split(), str2.split()
    
    # Generate all possible word orderings (rotations) for comparison
    permutations1 = [" ".join(p) for p in itertools.permutations(words1)]
    permutations2 = [" ".join(p) for p in itertools.permutations(words2)]
    
    # Compute the max similarity considering all orderings
    max_similarity = max(Levenshtein.normalized_similarity(p1, p2) for p1 in permutations1 for p2 in permutations2)
    
    return max_similarity

In [40]:
def geocoding_labelled_reports(df_labelled_chat) : 
    df_labelled_chat_geo = pd.DataFrame(columns=df_labelled_chat.columns.tolist()+['latitude', 'longitude'])
    for index, row in df_labelled_chat.iterrows():
        country = row["country"]
        if row.city:
            finest_loc_id = "city"
            finest_loc_vals = row.city
        elif row.region:
            finest_loc_id = "region"
            finest_loc_vals = row.region
        else:
            finest_loc_id = None
            print("No location information for response: ", response_row.appealCode)

        if finest_loc_id:
            # nominatim_query = {
            #     "country": country,
            #     finest_loc_id: finest_loc_vals
            # }
            time_last_request = time.time()
            nominatim_query=finest_loc_vals+", "+country

            time_new_request = time.time()
            if time_new_request - time_last_request < 5:
                time.sleep(time_new_request - time_last_request)
            nominatim_result = geolocator.geocode(nominatim_query, exactly_one=True, language="en")
            time_last_request = time_new_request
            if nominatim_result is None:
                print("No results for query: ", nominatim_query)
            else:      
                ## Double check that the known information is similar to the one found 
                # similarity = Levenshtein.normalized_similarity(finest_loc_vals, nominatim_result.raw['name'])
                similarity = rotated_levenshtein_similarity(finest_loc_vals, nominatim_result.raw['name'])
    
                if similarity > 0.7 : 
                    print("CORRECT location "+nominatim_result.raw['name']+" for "+finest_loc_vals+", similarity = "+str(similarity))
                    row["latitude"] = nominatim_result.latitude
                    row["longitude"] = nominatim_result.longitude
                    row[finest_loc_id] = nominatim_result.raw['name']
                    
                    #Find in case the location is a city
                    if finest_loc_id== "city" : 
                        #Reverse geocoding to get region/state name 
                        location_details = geolocator.reverse((nominatim_result.latitude, nominatim_result.longitude), exactly_one=True, addressdetails=True)
                        address = location_details.raw.get("address", {})
                        region = address.get("region")
                        if region is None : 
                            region = address.get("state")
                        row["region"] = region  

                    #Add the row to the DataFrame with geocoding
                else : 
                    print("WRONG location "+nominatim_result.raw['name']+" for "+finest_loc_vals+", similarity = "+str(similarity))
    return df_labelled_chat

In [41]:
df_labelled_chat_geo = geocoding_labelled_reports(df_labelled_chat)

WRONG location Bousssaada - Tizi N'Bchar for Bchar, similarity = 0.19999999999999996
WRONG location Route Ain Sefra-ElBayadh for Elbayadh, similarity = 0.29166666666666663
CORRECT location Beni Abbes for Beni Abbes, similarity = 1.0
CORRECT location Tamanrasset for Tamanrasset, similarity = 1.0
CORRECT location Tiaret for Tiaret, similarity = 1.0
CORRECT location Tinduf for Tindouf, similarity = 0.8571428571428572
CORRECT location Naâma for Naama, similarity = 0.8
CORRECT location Balochistan for Balochistan , similarity = 1.0


GeocoderUnavailable: HTTPSConnectionPool(host='nominatim.openstreetmap.org', port=443): Max retries exceeded with url: /search?q=Sindh+%2C+Pakistan&format=json&limit=1&accept-language=en (Caused by ReadTimeoutError("HTTPSConnectionPool(host='nominatim.openstreetmap.org', port=443): Read timed out. (read timeout=1)"))